# YouTube Dislikes Dataset Analysis
**Business Problem Statement:**
Analyze YouTube engagement metrics to understand the dynamics of user preference, predict the risk of negative reception (dislikes), and cluster content to optimize revenue generation based on demand.

**Economic Concepts Applied:**
1. **Demand-Supply / Revenue Optimization:** Video views, likes, and comments act as proxies for "consumer demand." By identifying high-demand clusters (using K-Means), content creators and platforms can optimize ad-placement strategies and supply more of what consumers want, thus maximizing revenue.
2. **Risk Analysis:** A high number of dislikes relative to likes poses a reputation risk. Predicting this risk (using Logistic Regression) allows for proactive management of brand safety and potential ad demonetization.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

## 1. Data Loading and Preprocessing

In [ ]:
# Load the dataset
file_path = '/Users/bankimkamila/.cache/kagglehub/datasets/dmitrynikolaev/youtube-dislikes-dataset/versions/2/youtube_dislike_dataset.csv'
df = pd.read_csv(file_path)

print("Initial Data Shape:", df.shape)

# Drop rows where 'comments' are null (minor data loss, ensures clean text if needed later)
df.dropna(subset=['comments'], inplace=True)

# Convert published_at to datetime
df['published_at'] = pd.to_datetime(df['published_at'])

# Feature Engineering: Calculate engagement ratios
# A high dislike ratio is a sign of poor market reception (High Risk)
df['total_interactions'] = df['likes'] + df['dislikes']
df['dislike_ratio'] = np.where(df['total_interactions'] > 0, df['dislikes'] / df['total_interactions'], 0)
df['engagement_rate'] = np.where(df['view_count'] > 0, (df['likes'] + df['comments'] + df['dislikes']) / df['view_count'], 0)

# Create a binary target for Logistic Regression (Risk Analysis)
# Let's say a dislike ratio > 5% is considered "High Risk" of poor reception.
threshold = 0.05
df['High_Risk'] = (df['dislike_ratio'] > threshold).astype(int)

print("Data Cleaning Complete. Missing values:")
print(df.isnull().sum())

## 2. Exploratory Data Analysis (EDA)

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df['dislike_ratio'], bins=50, kde=True)
plt.title('Distribution of Dislike Ratio (Risk Indicator)')
plt.xlabel('Dislike Ratio')
plt.ylabel('Frequency')
plt.savefig('dislike_ratio_dist.png')
plt.show()

# Business Interpretation: Most videos have a very low dislike ratio, indicating generally positive market reception. 
# The long tail indicates outliars that pose a brand severity risk.

# Correlation heatmap of numerical features
numerical_cols = ['view_count', 'likes', 'dislikes', 'comment_count', 'dislike_ratio']
plt.figure(figsize=(8, 6))
sns.heatmap(df[numerical_cols].corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix of Engagement Metrics')
plt.savefig('correlation_matrix.png')
plt.show()

# Business Interpretation: Views, likes, and comments are highly correlated, showing that overall engagement moves together (High Demand).
# Dislikes are also positively correlated with views, meaning viral videos naturally attract more detractors. 
# However, the dislike ratio isolates the sentiment risk regardless of volume.

## 3. Implementation of K-Means Algorithm (Demand Segmentation)
We cluster the videos based on `view_count`, `likes`, and `comment_count` to find "Demand Tiers".

In [ ]:
# Select features for clustering
features_cluster = df[['view_count', 'likes', 'comment_count']]

# Scale the features
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features_cluster)

# Apply K-Means
kmeans = KMeans(n_clusters=3, random_state=42)
df['Demand_Cluster'] = kmeans.fit_predict(scaled_features)

# Analyze the clusters
cluster_analysis = df.groupby('Demand_Cluster')[['view_count', 'likes', 'comment_count']].mean()
print("\nCluster Analysis (Mean values):")
print(cluster_analysis)

# Business Interpretation:
# Cluster 0: "Low/Niche Demand" - Low views and engagement. Represents the long tail of content.
# Cluster 1: "Viral/High Demand" - Massive views and likes. These videos represent the highest revenue-generating assets.
# Cluster 2: "Steady/Medium Demand" - Solid performance, reliable revenue stream.
# Optimizing supply means studying Cluster 1 characteristics to replicate success.

## 4. Implementation of Logistic Regression (Risk Prediction)
We predict whether a video is 'High_Risk' (Dislike Ratio > 5%) based on its view count and comment count.
Why? High views with high comments but disproportionate dislikes indicates controversial content that might scare away advertisers.

In [ ]:
# Define features (X) and target (y)
# We use log transformation to handle the massive skewness in views and comments
X = df[['view_count', 'comment_count']]
X = np.log1p(X) # log(1+x) transformation
y = df['High_Risk']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler_lr = StandardScaler()
X_train_scaled = scaler_lr.fit_transform(X_train)
X_test_scaled = scaler_lr.transform(X_test)

# Train the model
log_model = LogisticRegression(class_weight='balanced', random_state=42)
log_model.fit(X_train_scaled, y_train)

# Predictions
y_pred = log_model.predict(X_test_scaled)

# Evaluation
print("\nLogistic Regression Evaluation (Risk Prediction):")
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

# Business Interpretation:
# The model attempts to flag videos that are likely to have a high dislike ratio based purely on volume metrics (views and comments).
# If a video is predicted as High Risk, platforms can dynamically adjust ad-bidding strategies (Pricing Strategy) 
# or demonetize it preemptively to protect brand safe supply.